# 2 — Buck-Boost Controller Design

> **Goal.** Use the $G_{vd}(s)$ plant from notebook 1 to size a
> voltage-mode compensator that respects the RHP zero, then **prove**
> it works by running a switched closed-loop simulation in pure
> Python with a $v_{ref}$ step.

**Prerequisites**

- Buck-boost modeling notebook (`01_buck_boost_modeling.ipynb`).
- Buck and boost controller notebooks for context on K-factor
  Type-III sizing.

The recipe is the same as the boost's; what differs is the
$f_{z,RHP}$ location, which sets the bandwidth ceiling.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from buck_boost_model import (
    BuckBoostParams, control_to_output_tf, operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

params = BuckBoostParams()
print(operating_point_report(params))


## 1. Bandwidth target

For the default parameters ($D = 0.5$), $f_{z,RHP} \\approx 9.5$ kHz.
Target $f_c = f_{z,RHP} / 5 \\approx 1.9$ kHz with PM = 60°.


In [ ]:
Gvd = control_to_output_tf(params)
V_ramp = 5.0
plant = signal.TransferFunction(np.array(Gvd.num)/V_ramp, np.array(Gvd.den))

f_c_target = params.f_z_rhp / 5.0
pm_target = 60.0
print(f"RHP zero:   f_z = {params.f_z_rhp:7.0f} Hz")
print(f"Target:     f_c = {f_c_target:.0f} Hz, PM = {pm_target}°")


## 2. K-factor Type-III sizing

Identical algorithm to the boost notebook §4. Repeated here for
self-containment.


In [ ]:
def design_type3_kfactor(plant, f_c, pm_target):
    omega_c = 2 * np.pi * f_c
    _, _, ph_plant = signal.bode(plant, w=[omega_c])
    # scipy wraps phase to [-180°, +180°], but for a non-minimum-phase
    # plant (RHP zero + LC double pole) the true phase at f_c can dip
    # below -180°. Detect the wrap and unfold.
    ph_at_fc = ph_plant[0]
    if ph_at_fc > 0:
        ph_at_fc -= 360.0
    phi_lead = pm_target - 90.0 - ph_at_fc
    phi_lead = float(np.clip(phi_lead, 10.0, 175.0))
    phi_pair = phi_lead / 2
    k = np.tan(np.deg2rad(phi_pair/2 + 45.0)) ** 2
    omega_z = omega_c / np.sqrt(k)
    omega_p = omega_c * np.sqrt(k)
    num0 = np.polymul([1.0, omega_z], [1.0, omega_z])
    den0 = np.polymul([1.0, 0.0], np.polymul([1.0, omega_p], [1.0, omega_p]))
    open0 = signal.TransferFunction(
        np.polymul(num0, plant.num), np.polymul(den0, plant.den)
    )
    _, mag0, _ = signal.bode(open0, w=[omega_c])
    K = 10.0 ** (-mag0[0] / 20)
    return signal.TransferFunction(K * num0, den0), omega_z, omega_p, float(K)


Gc, omega_z, omega_p, K_dc = design_type3_kfactor(plant, f_c_target, pm_target)
print(f"Designed compensator:")
print(f"  zeros at  f_z = {omega_z/(2*np.pi):8.1f} Hz  (double)")
print(f"  HF poles  f_p = {omega_p/(2*np.pi):8.1f} Hz  (double)")
print(f"  DC gain K     = {K_dc:.4g}")


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2*np.pi*f
T_open = signal.TransferFunction(np.polymul(Gc.num, plant.num),
                                   np.polymul(Gc.den, plant.den))
_, mag_T, ph_T = signal.bode(T_open, w=w)
_, mag_p, ph_p = signal.bode(plant, w=w)
idx_cross = np.argmin(np.abs(mag_T))
f_cross = f[idx_cross]
pm = 180 + ph_T[idx_cross]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_mag.semilogx(f, mag_p, "C0--", alpha=0.5, label="Plant + $k_{PWM}$")
ax_mag.semilogx(f, mag_T, "C3", linewidth=2, label="Loop gain $T(s)$")
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_mag.axvline(f_cross, color="g", linestyle=":", alpha=0.5,
               label=f"$f_c$ ≈ {f_cross:.0f} Hz")
ax_mag.axvline(params.f_z_rhp, color="C3", linestyle=":", alpha=0.4,
               label="$f_{z,RHP}$")
ax_ph.semilogx(f, ph_p, "C0--", alpha=0.5)
ax_ph.semilogx(f, ph_T, "C3", linewidth=2)
ax_ph.axhline(-180, color="k", linestyle=":", alpha=0.3)
ax_ph.axvline(f_cross, color="g", linestyle=":", alpha=0.5)
ax_mag.set_ylabel("Magnitude [dB]"); ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]"); ax_mag.legend(loc="best", fontsize=8)
ax_mag.set_title(f"Compensated buck-boost loop: $f_c$ = {f_cross:.0f} Hz, "
                 f"PM = {pm:.1f}°")
plt.tight_layout()
plt.show()

print(f"Achieved:  f_c = {f_cross:.0f} Hz, PM = {pm:.1f}°")


## 3. Discretization

Tustin / bilinear at $T_s = 1/f_{sw}$.


In [ ]:
T_s = 1.0 / params.f_sw
Gc_d_num, Gc_d_den, _ = signal.cont2discrete((Gc.num, Gc.den), dt=T_s, method="bilinear")
b = np.asarray(Gc_d_num).flatten() / Gc_d_den[0]
a = np.asarray(Gc_d_den) / Gc_d_den[0]
print(f"Sample period T_s = {T_s*1e6:.3f} µs")
print()
print("Discrete-time recurrence (a[0] = 1):")
for i, bi in enumerate(b): print(f"  b[{i}] = {bi:+.6f}")
for i, ai in enumerate(a): print(f"  a[{i}] = {ai:+.6f}")


## 4. Switched-model closed-loop simulation

Pure-Python forward-Euler switched buck-boost + discretized compensator
running once per switching period (sample-and-hold). Warm-start at the
operating point so we test the small-signal response the compensator
was sized for (the cold-start of a buck-boost is dominated by
nonlinear inrush — not what the controller design assumes).

**A small twist vs the boost.** In the buck-boost the inductor pumps
charge into a NEGATIVE-polarity output cap. The switched model is:

```
ON (S closed, D off):
    L · di/dt = V_g
    C · dv_o/dt = -v_o / R   (cap drains through load)

OFF (S open, D on, in CCM with i_L > 0):
    L · di/dt = -v_o          (cap voltage opposes inductor)
    C · dv_o/dt = i_L - v_o/R (cap charges, load also drains)
```

We model $v_o$ as a positive magnitude and just track that.


In [ ]:
def simulate_closed_loop_buck_boost(
    params,
    b: np.ndarray, a: np.ndarray,
    *,
    t_end: float = 30e-3,
    t_step: float = 5e-3,
    v_ref_initial: float = 12.0,
    v_ref_final: float = 13.0,
    V_ramp: float = 5.0,
    samples_per_period: int = 200,
    warm_start: bool = True,
):
    '''Forward-Euler switched buck-boost + digital compensator.

    States are inductor current i_L and output cap voltage MAGNITUDE
    v_o. The compensator runs once per switching period (sample-and-
    hold), generating the duty for the next period.

    `warm_start=True` pre-loads i_L, v_o, and the compensator state at
    the operating point corresponding to v_ref_initial.
    '''
    T_s = 1.0 / params.f_sw
    dt_sim = T_s / samples_per_period
    n_steps = int(t_end / dt_sim) + 1

    n_state = len(a) - 1
    state = np.zeros(n_state)

    if warm_start:
        D_init = v_ref_initial / (v_ref_initial + params.V_g)
        I_L_avg = v_ref_initial / (params.R * (1.0 - D_init))
        # Inductor current at the BEGINNING of an ON interval = average
        # minus half the peak-to-peak ripple. Starting at the average
        # injects a half-period of extra charge into the cap on cycle 0
        # and causes the loop to drift before the controller catches up.
        T_s_period = 1.0 / params.f_sw
        delta_i_pp = params.V_g * D_init * T_s_period / params.L
        i_L = I_L_avg - delta_i_pp / 2.0
        v_o = v_ref_initial
        duty = D_init
        v_c_ss = duty * V_ramp
        for k in range(n_state):
            state[k] = -np.sum(a[k+1:]) * v_c_ss
    else:
        i_L = 0.0
        v_o = 0.0
        duty = 0.5

    record_every = max(1, samples_per_period // 50)
    n_rec = n_steps // record_every + 1
    t_hist = np.zeros(n_rec)
    v_o_hist = np.zeros(n_rec)
    i_L_hist = np.zeros(n_rec)
    duty_hist = np.zeros(n_rec)
    v_ref_hist = np.zeros(n_rec)
    rec_idx = 0

    for i in range(n_steps):
        t = i * dt_sim
        v_ref = v_ref_initial if t < t_step else v_ref_final

        cycle_pos_int = i % samples_per_period
        if cycle_pos_int == 0:
            err = v_ref - v_o
            v_c = b[0] * err + state[0]
            new_state = np.zeros_like(state)
            for j in range(n_state - 1):
                new_state[j] = b[j+1] * err - a[j+1] * v_c + state[j+1]
            new_state[n_state - 1] = b[n_state] * err - a[n_state] * v_c
            state = new_state
            duty = float(np.clip(v_c / V_ramp, 0.05, 0.92))

        switch_on = (cycle_pos_int / samples_per_period) < duty

        # Switched buck-boost ODE (positive-magnitude convention for v_o)
        if switch_on:
            v_L = params.V_g       # inductor sees V_g
            i_C = -v_o / params.R  # cap drains
        else:
            # OFF: diode conducts only if i_L > 0 (CCM); DCM detection:
            if i_L > 0:
                v_L = -v_o
                i_C = i_L - v_o / params.R
            else:
                v_L = 0.0          # diode off (DCM)
                i_C = -v_o / params.R
        i_L += (v_L / params.L) * dt_sim
        i_L = max(i_L, 0.0)
        v_o += (i_C / params.C) * dt_sim

        if i % record_every == 0 and rec_idx < n_rec:
            t_hist[rec_idx] = t
            v_o_hist[rec_idx] = v_o
            i_L_hist[rec_idx] = i_L
            duty_hist[rec_idx] = duty
            v_ref_hist[rec_idx] = v_ref
            rec_idx += 1

    return {
        "t": t_hist[:rec_idx], "v_o": v_o_hist[:rec_idx],
        "i_L": i_L_hist[:rec_idx], "duty": duty_hist[:rec_idx],
        "v_ref": v_ref_hist[:rec_idx],
    }


In [ ]:
sim = simulate_closed_loop_buck_boost(
    params, b=b, a=a,
    t_end=30e-3, t_step=5e-3,
    v_ref_initial=12.0, v_ref_final=13.0,
    V_ramp=V_ramp, warm_start=True,
)

print(f"Simulated {len(sim['t'])} samples over {sim['t'][-1]*1e3:.2f} ms")
pre_mask  = (sim['t'] > 4.0e-3) & (sim['t'] < 5.0e-3)
post_mask = sim['t'] > 27.0e-3
print()
print(f"Pre-step  |V_o| (mean 4-5 ms):   {np.mean(sim['v_o'][pre_mask]):.4f} V "
      f"(target 12.0)")
print(f"Post-step |V_o| (mean 27-30 ms): {np.mean(sim['v_o'][post_mask]):.4f} V "
      f"(target 13.0)")
D_pre = 12.0 / (12.0 + params.V_g)
D_post = 13.0 / (13.0 + params.V_g)
print(f"Pre-step duty:  {np.mean(sim['duty'][pre_mask]):.4f}  "
      f"(expect {D_pre:.4f})")
print(f"Post-step duty: {np.mean(sim['duty'][post_mask]):.4f}  "
      f"(expect {D_post:.4f})")


In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axs[0].plot(sim['t']*1e3, sim['v_o'], 'C0', linewidth=0.8, label="$|v_o|$ (switched)")
axs[0].plot(sim['t']*1e3, sim['v_ref'], 'C3--', linewidth=2, label="$v_{ref}$")
axs[0].axvline(5.0, color="k", linestyle=":", alpha=0.4, label="step")
axs[0].set_ylabel("|Output voltage| [V]")
axs[0].set_title("Closed-loop buck-boost (warm-start at OP): step "
                 "$v_{ref}$ 12 V → 13 V at $t$ = 5 ms")
axs[0].legend(loc="lower right")

axs[1].plot(sim['t']*1e3, sim['i_L'], 'C1', linewidth=0.8)
axs[1].axvline(5.0, color="k", linestyle=":", alpha=0.4)
i_L_pre = 12.0 / (params.R * (1 - D_pre))
i_L_post = 13.0 / (params.R * (1 - D_post))
axs[1].axhline(i_L_pre, color="k", linestyle=":", alpha=0.3,
               label=f"pre-step $I_L$ = {i_L_pre:.2f} A")
axs[1].axhline(i_L_post, color="r", linestyle=":", alpha=0.3,
               label=f"post-step $I_L$ = {i_L_post:.2f} A")
axs[1].set_ylabel("Inductor current [A]")
axs[1].legend(loc="lower right")

axs[2].plot(sim['t']*1e3, sim['duty'], 'C2', linewidth=1.0)
axs[2].axvline(5.0, color="k", linestyle=":", alpha=0.4)
axs[2].axhline(D_pre, color="k", linestyle=":", alpha=0.3,
               label=f"pre-step D = {D_pre:.3f}")
axs[2].axhline(D_post, color="r", linestyle=":", alpha=0.3,
               label=f"post-step D = {D_post:.3f}")
axs[2].set_ylabel("Duty cycle")
axs[2].legend(loc="lower right")

axs[3].plot(sim['t']*1e3, sim['v_ref'] - sim['v_o'], 'C4', linewidth=0.8)
axs[3].axvline(5.0, color="k", linestyle=":", alpha=0.4)
axs[3].axhline(0, color="k", linestyle=":", alpha=0.3)
axs[3].set_ylabel("Tracking error\n$v_{ref} - |v_o|$ [V]")
axs[3].set_xlabel("Time [ms]")

plt.tight_layout()
plt.show()


In [ ]:
mask_after = sim['t'] > 5.0e-3
t_after = sim['t'][mask_after] - 5.0e-3
v_o_after = sim['v_o'][mask_after]

overshoot_pct = (np.max(v_o_after) - 13.0) / (13.0 - 12.0) * 100
dip_amount = 12.0 - np.min(v_o_after)
settled = np.abs(v_o_after - 13.0) < 0.02 * (13.0 - 12.0)
unsettled = np.where(~settled)[0]
settling_ms = t_after[
    min(unsettled[-1] + 1, len(t_after) - 1) if len(unsettled) else 0
] * 1e3
v_o_10 = 12.0 + 0.1
v_o_90 = 12.0 + 0.9
rise_start = np.argmax(v_o_after >= v_o_10)
rise_end = np.argmax(v_o_after >= v_o_90)
rise_time_ms = (t_after[rise_end] - t_after[rise_start]) * 1e3

ss_error = 13.0 - np.mean(sim['v_o'][sim['t'] > 27e-3])

print("Closed-loop step-response metrics ($v_{ref}$: 12 → 13 V)")
print(f"  Initial dip (RHP zero)     = {dip_amount * 1e3:7.1f} mV below pre-step")
print(f"  Rise time (10% → 90%)      = {rise_time_ms:7.3f} ms")
print(f"  Peak overshoot             = {overshoot_pct:7.2f} %")
print(f"  Settling time (±2 %)       = {settling_ms:7.3f} ms")
print(f"  Steady-state error         = {ss_error*1e3:+7.2f} mV "
      f"({ss_error / 13.0 * 100:+.3f} %)")
print()
# Same generous gate as the boost — RHP-zero-limited loops are
# fundamentally slow.
if abs(ss_error) < 0.15 and overshoot_pct < 50 and settling_ms < 40.0:
    print("✅  Closed-loop controller PROVEN on the switched buck-boost:")
    print(f"    • SS error    = {ss_error*1e3:.1f} mV ({ss_error/13.0*100:.2f} %)")
    print(f"    • Overshoot   = {overshoot_pct:.1f} %")
    print(f"    • Settling    = {settling_ms:.1f} ms")
    print(f"    • RHP dip     = {dip_amount*1e3:.1f} mV (visible)")
    print()
    print(f"    Compare with buck: 1.4 ms settling, 0.6 % overshoot.")
    print(f"    Buck-boost is ~{settling_ms/1.4:.0f}× slower because the RHP zero caps")
    print(f"    the loop bandwidth. Same fundamental limit as the boost.")
else:
    print("⚠️   Closed-loop response off-target — revisit f_c or PM.")


## 5. Summary

You sized a Type-III compensator for the inverting buck-boost
respecting the $f_c \\le f_{z,RHP}/5$ ceiling, discretized it via
Tustin, and ran a switched closed-loop simulation that demonstrates:

- Stable tracking of a step in $v_{ref}$.
- The characteristic RHP-zero dip before recovery.
- Slow settling vs the buck — fundamentally capped by the topology.

**Suggested exercises**

1. Re-tune for $V_o = 24$ V (D = 0.67). Verify the new $f_{z,RHP}$
   is lower and the closed loop is correspondingly slower.
2. Add a 50 % load step ($R$: 12 → 6 Ω) at $t = 20$ ms. Watch the
   output sag and the controller recover — the buck-boost's load
   regulation is fundamentally limited by the same RHP zero.
3. Build a SEPIC (4th-order non-inverting buck-boost). Where do the
   TWO RHP zeros come from? How does the closed-loop bandwidth limit
   compare?
